# Task 4 — Loan Approval Prediction (Kaggle)

**Description:** This notebook builds a binary classifier to predict whether a loan application is approved. It focuses on handling missing values, encoding categorical features, and evaluating models on imbalanced data using precision, recall, and F1-score.

**Dataset (Kaggle):** Loan Approval Prediction Dataset
- Link: https://www.kaggle.com/datasets/architsharma01/loan-approval-prediction-dataset
- File path: `/kaggle/input/datasets/architsharma01/loan-approval-prediction-dataset/loan_approval_dataset.csv`

**What this notebook covers:**
- Dataset inspection (rows, columns, missing values, class balance)
- Handling missing values and encoding categorical features
- Logistic Regression vs. Decision Tree
- Imbalanced data evaluation using precision, recall, and F1-score
- Bonus: SMOTE to address class imbalance


### Alt Step 1 — Import libraries

In [ ]:
# 1. Import libraries
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTE


### Alt Step 2 — Load dataset

In [ ]:
# 2. Load dataset (Kaggle path)
df = pd.read_csv(
    "/kaggle/input/datasets/architsharma01/loan-approval-prediction-dataset/loan_approval_dataset.csv"
)

print("Dataset Shape:", df.shape)
print(df.head())


### Alt Step 3 — Data cleaning

In [ ]:
# 3. Data cleaning
# Trim spaces in column names
df.columns = df.columns.str.strip()

# Drop unnecessary column
df = df.drop("loan_id", axis=1)

# Handle missing values (drop rows with NA)
df = df.dropna()


### Alt Step 4 — Encoding categorical features

In [ ]:
# 4. Encoding
le = LabelEncoder()

categorical_cols = ["education", "self_employed", "loan_status"]

for col in categorical_cols:
    df[col] = le.fit_transform(df[col])


### Alt Step 5 — Split data

In [ ]:
# 5. Split data
X = df.drop("loan_status", axis=1)
y = df["loan_status"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)


### Alt Step 6 — Feature scaling

In [ ]:
# 6. Feature scaling
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


### Alt Step 7 — Handle imbalance (SMOTE)

In [ ]:
# 7. Handle imbalance (SMOTE)
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

print("After SMOTE:", np.bincount(y_train))


### Alt Step 8 — Train models

In [ ]:
# 8. Model training
# Logistic Regression
lr = LogisticRegression(max_iter=2000)
lr.fit(X_train, y_train)

# Decision Tree
dt = DecisionTreeClassifier(max_depth=7, min_samples_split=10, random_state=42)
dt.fit(X_train, y_train)

# Random Forest
rf = RandomForestClassifier(n_estimators=150, random_state=42)
rf.fit(X_train, y_train)


### Alt Step 9 — Evaluate models

In [ ]:
# 9. Evaluation
def evaluate_model(name, model):
    y_pred = model.predict(X_test)
    print(f"\n=== {name} ===")
    print(classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

evaluate_model("Logistic Regression", lr)
evaluate_model("Decision Tree", dt)
evaluate_model("Random Forest", rf)


### Alt Step 10 — Prediction function

In [ ]:
# 10. Prediction function
def predict_loan(model, scaler, input_data):
    input_df = pd.DataFrame([input_data])

    # Encode categorical manually
    input_df["education"] = input_df["education"].map({"Graduate": 1, "Not Graduate": 0})
    input_df["self_employed"] = input_df["self_employed"].map({"Yes": 1, "No": 0})

    # Scale
    input_scaled = scaler.transform(input_df)

    # Predict
    prediction = model.predict(input_scaled)

    return "Approved ✅" if prediction[0] == 1 else "Rejected ❌"


### Alt Step 11 — Test inputs

In [ ]:
# 11. Test inputs
# Example 1 (Approved)
sample_input = {
    "no_of_dependents": 1,
    "education": "Graduate",
    "self_employed": "No",
    "income_annum": 800000,
    "loan_amount": 200000,
    "loan_term": 12,
    "cibil_score": 780,
    "residential_assets_value": 500000,
    "commercial_assets_value": 200000,
    "luxury_assets_value": 100000,
    "bank_asset_value": 300000,
}

print("Prediction:", predict_loan(rf, scaler, sample_input))

# Example 2 (Rejected)
sample_input2 = {
    "no_of_dependents": 4,
    "education": "Not Graduate",
    "self_employed": "Yes",
    "income_annum": 200000,
    "loan_amount": 500000,
    "loan_term": 36,
    "cibil_score": 500,
    "residential_assets_value": 50000,
    "commercial_assets_value": 20000,
    "luxury_assets_value": 10000,
    "bank_asset_value": 30000,
}

print("Prediction:", predict_loan(rf, scaler, sample_input2))


de